<!-- course-header -->
# Lesson 10: Working with LLM APIs

*Companion notebook for lesson 10. Read the lesson [online](https://ashutoshjoshi1.github.io/AI-ZerotoMastery/lessons/10-llm-apis.html) (it works well on a phone) or [offline](../lessons/10-llm-apis.html). Worked solutions are in `solutions/10-llm-apis-solutions.ipynb`; try each exercise first.*

<a href="https://colab.research.google.com/github/ashutoshjoshi1/AI-ZerotoMastery/blob/main/notebooks/10-llm-apis.ipynb"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"></a>

Call hosted and local models from code like a professional, covering messages and system prompts, streaming, reliable structured output, errors and retries, token costs, caching, batches and concurrency, then build a streaming command-line chatbot.

**Runs on:** Any computer. Cells that call a hosted model are marked and need an API key; the helpers and the fake-client exercises run offline.

**How to use it:** read a section of the lesson, then run the matching cells here, in order. Each exercise has a cell titled *Hands-on* with the same name as in the lesson. Change numbers, break things on purpose, and note what happens in your learning log. Cells that need an API key, a GPU or a download say so in their first line.

## Setup

Install what this lesson needs. Skip this cell if your environment from lesson 00 already has these packages.

In [ ]:
%pip install -q anthropic openai pydantic python-dotenv

In [ ]:
import os
from dotenv import load_dotenv
import anthropic

load_dotenv()   # reads ANTHROPIC_API_KEY from a .env file next to this notebook, if there is one
# On Google Colab, add the key under the key icon (Secrets), then uncomment the next two lines:
# from google.colab import userdata
# os.environ["ANTHROPIC_API_KEY"] = userdata.get("ANTHROPIC_API_KEY")

client = anthropic.Anthropic()
MODEL = "claude-sonnet-5-5"          # check your provider's current model list
print("API key found" if os.environ.get("ANTHROPIC_API_KEY") else
      "No API key set: cells marked 'Needs an API key' will fail until you add one")

## The shape of every chat API

This lesson uses the Anthropic Python SDK for the main examples, with the OpenAI-style equivalent alongside, because that second format is also what most local and open-model servers speak. The concepts transfer to any provider.

Run in a terminal:

```bash
uv add anthropic openai pydantic python-dotenv
```

> **Model names change**
>
> Providers release new models every few months. The examples use `claude-sonnet-5-5` as a capable default and `claude-haiku-5-5` as a fast, cheap one. Check your provider's model list and keep the name in one configuration constant so swapping models is a one-line change.

## First calls

**Option: Anthropic**

In [ ]:
# Needs an API key (see the setup cell)
import anthropic
from dotenv import load_dotenv

load_dotenv()                                   # ANTHROPIC_API_KEY from .env
client = anthropic.Anthropic()
MODEL = "claude-sonnet-5-5"

resp = client.messages.create(
    model=MODEL,
    max_tokens=2000,                            # upper bound on the reply length
    system="You are a concise teaching assistant for an AI engineering course.",
    messages=[{"role": "user", "content": "Explain overfitting in two sentences."}],
)
for block in resp.content:                      # a list of blocks: text, tool calls, thinking...
    if block.type == "text":
        print(block.text)
print(resp.stop_reason)                         # "end_turn", or "max_tokens" if it was cut off
print(resp.usage.input_tokens, resp.usage.output_tokens)

**Option: OpenAI-compatible**

In [ ]:
# Needs an API key (see the setup cell)
from openai import OpenAI
from dotenv import load_dotenv

load_dotenv()                                   # OPENAI_API_KEY from .env
client = OpenAI()
resp = client.chat.completions.create(
    model="YOUR_MODEL_NAME",
    max_tokens=2000,
    messages=[
        {"role": "system", "content": "You are a concise teaching assistant."},
        {"role": "user", "content": "Explain overfitting in two sentences."},
    ],
)
print(resp.choices[0].message.content)
print(resp.choices[0].finish_reason)            # "stop" or "length"
print(resp.usage.prompt_tokens, resp.usage.completion_tokens)

### Read the response by block type

The content of an Anthropic response is a *list* of blocks. Today it holds one text block; tomorrow it may hold a tool call or a thinking block from a reasoning model in front of the text. Code that reads `resp.content[0].text` breaks the day that happens. Read by type, and keep the helper around for the rest of the course:

In [ ]:
def text_of(resp) -> str:
    """Join the text blocks of a response. Never rely on resp.content[0]."""
    return "".join(b.text for b in resp.content if b.type == "text")

### Multi-turn conversations

Because the API is stateless, a conversation is a list you keep appending to.

In [ ]:
# Needs an API key (see the setup cell)
history = []

def chat(user_text: str) -> str:
    history.append({"role": "user", "content": user_text})
    resp = client.messages.create(model=MODEL, max_tokens=2000, messages=history,
                                  system="You are a helpful tutor.")
    reply = text_of(resp)
    history.append({"role": "assistant", "content": reply})
    return reply

print(chat("What is a learning rate?"))
print(chat("What happens if mine is too high?"))   # "mine" is understood because history was resent

### The parameters that matter

`max_tokens` is a hard cap on output length, and on models that think before answering the thinking counts towards it, so set it with room to spare and check `stop_reason`. `system` carries standing instructions. `output_config.effort` controls how much a reasoning model thinks (lesson 12). Many APIs and local servers accept a `temperature`; the current Claude models fix their sampling settings and reject other values, so steer with instructions and structured output instead.

Current Claude models reason before they answer by default, and the **effort** level is the dial. For a cheap classification you do not want a paragraph of thinking:

In [ ]:
# Needs an API key (see the setup cell)
resp = client.messages.create(
    model="claude-haiku-5-5", max_tokens=1000,
    output_config={"effort": "low"},            # little or no thinking: cheap and fast
    messages=[{"role": "user", "content": "One word, hardware or software: 'The fan is grinding.'"}],
)
print(text_of(resp), "|", resp.usage.output_tokens, "output tokens")

### Hands-on: a provider-neutral ask() helper

Build the small wrapper that the rest of your code will call, and test it without spending a token.

1. Write `ask(messages, system=None, max_tokens=1000, client=client)` that calls the Anthropic API and returns a tuple `(text, usage, stop_reason)` where `usage` is a plain dict `{"in": ..., "out": ...}`.
2. Write a `FakeAnthropic` class whose `.messages.create(**kwargs)` ignores the arguments and returns an object shaped like a real response (a list of blocks in `.content`, a `.stop_reason`, a `.usage` with token counts). `types.SimpleNamespace` is enough. Make the fake return two blocks, a thinking block and a text block, so your helper is forced to read by type.
3. Add an `ask_openai(...)` twin for the OpenAI-compatible format, and a `FakeOpenAI` for it.
4. You are done when both fakes make their helper return exactly `("Hello from the fake.", {"in": 12, "out": 5}, "end_turn")`, and, if you have a key, the real client returns a sensible answer through the same `ask()`.

In [ ]:
# Your code here

## Streaming

A long answer can take many seconds. Streaming sends tokens as they are generated, so users see text appear immediately. The **time to first token** is what makes an app feel fast.

In [ ]:
# Needs an API key (see the setup cell)
with client.messages.stream(
    model=MODEL, max_tokens=2000,
    messages=[{"role": "user", "content": "Explain gradient descent to a new engineer."}],
) as stream:
    for text in stream.text_stream:
        print(text, end="", flush=True)
    final = stream.get_final_message()               # full message with usage after streaming
print("\n", final.usage.output_tokens, "tokens")

## Structured output you can trust

### Define the shape with Pydantic

In [ ]:
from pydantic import BaseModel, Field
from typing import Literal

class Ticket(BaseModel):
    category: Literal["hardware", "software", "data", "other"]
    severity: int = Field(ge=1, le=5, description="1 = cosmetic, 5 = instrument down")
    instrument_id: str | None = Field(default=None, description="e.g. 'P-117' if mentioned")
    summary: str = Field(max_length=200)

### Ask the API to fill it in

Modern APIs accept a JSON schema as an **output format** and constrain generation so the reply is valid JSON that matches it. The Anthropic SDK's `parse` helper takes the Pydantic class, sends its schema, and validates the reply on the way back. The API guarantees the *structure* (names, types, enums, required fields); Pydantic then enforces the rest, such as the 1 to 5 range on `severity`, and raises a `pydantic.ValidationError` if the reply breaks it.

In [ ]:
# Needs an API key (see the setup cell)
def extract_ticket(email_text: str) -> Ticket:
    resp = client.messages.parse(
        model=MODEL,
        max_tokens=2000,
        output_format=Ticket,                                          # schema goes to the API
        messages=[{"role": "user", "content": f"Create a ticket from this email:\n\n{email_text}"}],
    )
    return resp.parsed_output                                          # a validated Ticket

t = extract_ticket("Hi, unit P-117 at the coastal site stopped logging overnight. "
                   "Power LED is on but no data since 02:00. Urgent, campaign starts Monday.")
print(t)
# category='hardware' severity=4 instrument_id='P-117' summary='Unit P-117 stopped logging ...'

> **Validate, then repair**
>
> If validation fails, send the error message back to the model ("Your output failed validation: severity must be at most 5. Fix it.") and retry once or twice. Libraries such as Instructor automate this loop.

### Hands-on: validate, then repair

Build the repair loop yourself, and test it against a scripted model so you can see it work without a key.

1. Write `extract_with_repair(email, llm, max_attempts=3) -> Ticket`. `llm(messages) -> str` is any function that returns the model's text. Ask for JSON matching `Ticket.model_json_schema()`, parse it with `Ticket.model_validate_json`, and on failure append the assistant's reply and a user message quoting the validation error, then call again.
2. Write `scripted(replies)`, which returns an `llm` that hands out the given replies in order. Script a first reply with `"severity": 9` and a second that is valid.
3. You are done when the scripted run returns a valid `Ticket` after exactly one repair (count the calls), a script that always returns `severity 9` makes the function raise after three attempts, and, with a key, the real model passes first time through the same function.

In [ ]:
# Your code here

## Errors, retries and rate limits

The official SDKs already retry transient errors with exponential backoff. Configure it, and handle what remains:

In [ ]:
# Needs an API key (see the setup cell)
client = anthropic.Anthropic(max_retries=4, timeout=60.0)

try:
    resp = client.messages.create(model=MODEL, max_tokens=1000,
                                  messages=[{"role": "user", "content": "Hello"}])
except anthropic.RateLimitError:
    ...   # back off further, queue the work, or route to another model
except anthropic.APIConnectionError:
    ...   # network trouble or timeout: alert if persistent
except anthropic.APIStatusError as e:
    print(e.status_code, e.message)   # bad request, auth, etc.: log and fix, do not retry blindly

A request can also succeed and still not give you an answer: `stop_reason` is `"max_tokens"` when the cap cut the reply short, and `"refusal"` when a safety system declined. Check it before you use the text.

For production, add a **fallback**: if the primary model keeps failing, try a second model or provider, and degrade gracefully rather than crashing.

In [ ]:
TRANSIENT = (anthropic.RateLimitError, anthropic.APIConnectionError, anthropic.InternalServerError)

def ask_with_fallback(messages, models=(MODEL, "claude-haiku-5-5"), **kwargs):
    last_error = None
    for model in models:
        try:
            return client.messages.create(model=model, max_tokens=1000, messages=messages, **kwargs)
        except TRANSIENT as e:            # the SDK already retried this model; move on
            last_error = e
    raise last_error

## Tokens, cost and speed

### Estimate before you build

Providers price input and output tokens separately, and output tokens cost several times more than input tokens. A back-of-envelope calculation before you build saves surprises:

In [ ]:
def monthly_cost(requests_per_day, in_tokens, out_tokens, price_in_per_m, price_out_per_m):
    per_request = in_tokens / 1e6 * price_in_per_m + out_tokens / 1e6 * price_out_per_m
    return per_request * requests_per_day * 30

# 5,000 requests a day, 3,000 tokens of prompt (instructions + retrieved context), 400 tokens out.
# Illustrative prices for a mid-size model at the time of writing; plug in your provider's current list:
print(monthly_cost(5_000, 3_000, 400, price_in_per_m=2.0, price_out_per_m=10.0))   # 1500.0 per month

Count tokens exactly with the provider's counting endpoint rather than guessing. Tokenisers differ between providers, so a count from another vendor's tokeniser library is wrong for Claude:

In [ ]:
# Needs an API key (see the setup cell)
long_text = "Overfitting happens when a model memorises noise in its training data. " * 200
count = client.messages.count_tokens(model=MODEL, messages=[{"role": "user", "content": long_text}])
print(count.input_tokens)

### Prompt caching

If many requests share a long prefix (a big system prompt, a manual, a set of examples), providers can cache the processed prefix. Cached input tokens are billed at a fraction of the normal price and processed faster. With the Anthropic API you mark where the cacheable prefix ends:

In [ ]:
import os
if not os.path.exists("instrument_manual.txt"):   # a stand-in manual so the cell runs; use a real one
    section = ("Section 4.2, replacing the filter wheel belt: power down, remove the cover, fit belt FW-BELT-02 "
               "and tighten the motor mount screws to 0.8 N·m. Section 5.1, desiccant: replace every six months, "
               "or every three months at humid sites. ")
    open("instrument_manual.txt", "w", encoding="utf-8").write(section * 150)   # several thousand tokens, enough to cache
manual = open("instrument_manual.txt").read()          # thousands of tokens, the same every call

def ask_manual(question: str) -> str:
    resp = client.messages.create(
        model=MODEL, max_tokens=1000,
        system=[
            {"type": "text", "text": "Answer using only the manual below. Quote section numbers."},
            {"type": "text", "text": manual, "cache_control": {"type": "ephemeral"}},
        ],
        messages=[{"role": "user", "content": question}],
    )
    u = resp.usage
    print("cache write", u.cache_creation_input_tokens, "cache read", u.cache_read_input_tokens)
    return text_of(resp)

In [ ]:
# Needs an API key (see the setup cell)
print(ask_manual("How tight should the motor mount screws be?"))   # first call writes the cache
print(ask_manual("How often is the desiccant replaced at humid sites?"))   # second call reads it

The first call writes the cache; calls within the cache lifetime read it. The cache is a **prefix match** (tools, then system, then messages), so put stable content first and variable content last; short prefixes are not cached at all; and if `cache_read_input_tokens` stays at zero, something in your prefix is silently changing.

### Concurrency and batch processing

To process thousands of documents, run requests concurrently, with a limit so you stay under your rate limits.

In [ ]:
import asyncio
from anthropic import AsyncAnthropic

aclient = AsyncAnthropic(max_retries=5)
sem = asyncio.Semaphore(8)                        # at most 8 requests in flight

async def summarise(doc: str) -> str:
    async with sem:
        r = await aclient.messages.create(model="claude-haiku-5-5", max_tokens=500,
                                          output_config={"effort": "low"},
                                          messages=[{"role": "user", "content": f"Summarise in one sentence:\n{doc}"}])
        return text_of(r)

async def main(docs):
    return await asyncio.gather(*(summarise(d) for d in docs))

# summaries = asyncio.run(main(docs))

In [ ]:
# Needs an API key (see the setup cell)
docs = ["The instrument stopped logging overnight after a failed upload and needed a reboot.",
        "Replace the desiccant every six months, or every three months at humid sites."]
summaries = await main(docs)     # in a notebook, use await instead of asyncio.run
print(summaries)

For large jobs that do not need answers immediately, use the **Batch API**: submit thousands of requests at once, collect the results within hours, at a substantial discount. Each request carries a `custom_id`, because results come back in any order:

In [ ]:
# Needs an API key (see the setup cell)
from anthropic.types.message_create_params import MessageCreateParamsNonStreaming
from anthropic.types.messages.batch_create_params import Request

tickets = ["The fan in the enclosure is grinding loudly", "After updating to 2.4 the GUI crashes",
           "Yesterday's ozone values are all exactly zero"]

requests = [
    Request(custom_id=f"ticket-{i}", params=MessageCreateParamsNonStreaming(
        model="claude-haiku-5-5", max_tokens=300, output_config={"effort": "low"},
        messages=[{"role": "user", "content": f"One word, hardware/software/data/other: {t}"}]))
    for i, t in enumerate(tickets)
]
batch = client.messages.batches.create(requests=requests)
print(batch.id, batch.processing_status)          # "in_progress" until every request has run

In [ ]:
# Needs an API key (see the setup cell)
import time

while client.messages.batches.retrieve(batch.id).processing_status != "ended":
    time.sleep(30)

results = {}
for item in client.messages.batches.results(batch.id):      # any order: key by custom_id
    if item.result.type == "succeeded":
        results[item.custom_id] = text_of(item.result.message)
    else:
        print(item.custom_id, "failed:", item.result.type)   # errored, canceled or expired
print(results)

## Local and open models

Many local servers (Ollama, vLLM, llama.cpp's server, LM Studio) expose an **OpenAI-compatible** endpoint, so the same client code works by changing the base URL.

Run in a terminal:

```bash
# install Ollama from ollama.com, then:
ollama pull llama3.2          # a small open model, a couple of GB
```

In [ ]:
# Needs an API key (or a local Ollama server running: see the text above)
from openai import OpenAI

local = OpenAI(base_url="http://localhost:11434/v1", api_key="ollama")   # the key is ignored locally
r = local.chat.completions.create(
    model="llama3.2",
    messages=[{"role": "user", "content": "Give me three uses of a spectrometer."}],
)
print(r.choices[0].message.content)

### Hands-on: a streaming CLI chatbot

Build `chat.py`, a terminal chatbot that:

1. Streams replies as they are generated.
2. Keeps conversation history, and trims the oldest turns when the history exceeds about 20,000 tokens (estimate with `count_tokens`, or roughly 4 characters per token).
3. Prints the running token usage and an estimated cost after each reply.
4. Supports `/reset` to clear history, `/system <text>` to change the system prompt, and `/quit`.
5. Survives a network error without losing the conversation.

Develop it against the `FakeAnthropic` from the first exercise (give it a `messages.stream` that yields a few words) and switch to the real client at the end. You are done when a ten-turn conversation shows the input-token count climbing each turn, and pulling the network cable mid-reply leaves the history intact.

In [ ]:
# Your code here

## Check yourself

Take the quiz at the end of the lesson page. Then close everything and write three or four sentences in your learning log about what you learned, from memory.